### Attention is all you need

The self-attention layer incorporates relevant information from previous positions that
help process the current token

![image.png](./49f78880_image.png)

Multiple `token positions` going into the attention layer; the final one is the one being
currently processed. The attention mechanism operates on
the input vector at that position. It incorporates relevant information from
the context into the vector it produces as the output for that position.

![image-2.png](./49f78880_image-2.png)

Two main steps are involved in the attention mechanism:
1. **relevance scoring**: A way to score how relevant each of the previous input tokens are
to the current token being processed.
2. **combining information**: Using those scores, we combine the information from the various
positions into a single output vector.

![image-3.png](./49f78880_image-3.png)

To give the Transformer more extensive attention capability, the attention
mechanism is duplicated and executed multiple times in `parallel`. Each of
these parallel applications of attention is conducted into an `attention head`.

![image-4.png](./49f78880_image-4.png)

Each attention head produces its own output vector for the current token position. These
vectors are then concatenated and linearly transformed to produce the final output
of the self-attention layer for that token position.

**How many heads?** Common choices are `8`, `12`, `16`, or `32` heads, depending on the model size.
With HF transformers library, you can check the number of heads with:

```python
 model.config.num_attention_heads
```

In [3]:
from transformers import pipeline
generator = pipeline('text-generation', model='Qwen/Qwen3-0.6B')
print(generator.model)

print(f"-"*40+"\n")

print(generator.model.config)

del generator

Device set to use cuda:0


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [6]:
#compare multiple models side-by-side
import io
from itertools import zip_longest
from transformers import AutoModelForCausalLM

_models = ["Qwen/Qwen3-0.6B", "google/gemma-3-270m"]  # N models

# 1. Load models and capture printed representation
model_strs = {}
config_strs = {}
for model_name in _models:
    model = AutoModelForCausalLM.from_pretrained(model_name)
    buf = io.StringIO()
    print(model, file=buf)
    model_strs[model_name] = buf.getvalue().strip().split("\n")  # list of lines
    buf = io.StringIO()
    print(model.config, file=buf)
    config_strs[model_name] = buf.getvalue().strip().split("\n")
    del model, buf

# 2. Compute max width for each model column
col_widths = []
for model_name in _models:
    lines = model_strs[model_name]
    max_len = max(len(l) for l in lines)
    col_widths.append(max_len + 4)  # padding

# 3. Print header row
header_cells = [
    model_name.ljust(col_widths[i])
    for i, model_name in enumerate(_models)
]
print(" | ".join(header_cells))

# 4. Print separator row
sep_cells = [
    "-" * col_widths[i]
    for i in range(len(_models))
]
print(" | ".join(sep_cells))

# 5. Print all rows side-by-side
# zip_longest creates rows of N columns
for row in zip_longest(*model_strs.values(), fillvalue=""):
    row_cells = [
        row[i].ljust(col_widths[i])
        for i in range(len(_models))
    ]
    print(" | ".join(row_cells))

# 6. Print separator row
sep_cells = [
    "-" * col_widths[i]
    for i in range(len(_models))
]
print(" | ".join(sep_cells))    

# 6. Print all rows side-by-side
# zip_longest creates rows of N columns
for row in zip_longest(*config_strs.values(), fillvalue=""):
    row_cells = [
        row[i].ljust(col_widths[i])
        for i in range(len(_models))
    ]
    print(" | ".join(row_cells))

Qwen/Qwen3-0.6B                                                                    | google/gemma-3-270m                                                              
---------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------
Qwen3ForCausalLM(                                                                  | Gemma3ForCausalLM(                                                               
  (model): Qwen3Model(                                                             |   (model): Gemma3TextModel(                                                      
    (embed_tokens): Embedding(151936, 1024)                                        |     (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)    
    (layers): ModuleList(                                                          |     (layers): ModuleList(                                                       

### Attention calculations

Inside a single attention head, the attention layer is processing attention
for a single position.

The `inputs` to the layer are:
- The vector representation of the current position or token
- The vector representations of the previous tokens

The goal is to produce a new representation of the current position that incorporates relevant information from the previous tokens:

For example, if we’re processing the last position in the
sentence “Max fed the cat (Marietto) because it,” we want “it” to
represent the cat—so attention bakes in “cat information”
from the cat token.

Three projection matrices are produced to create the components that interact in this calculation:
- A `query` projection matrix
- A `key` projection matrix
- A `value` projection matrix
These matrices are learned during training and are specific to each attention head.

![image.png](./39fc787c_image.png)

- Query (Q): what this token is `looking for`
- Key (K): what this token can be `matched by`
- Value (V): the `information each token carries` that can be `shared with others`

#### Q,K,V analogy

An analogy can be a **library**, where the query is the question asked, the key is the catalog reference of all books (current + previous), and the value is the content of all books (current + previous).

- Library search analogy
Imagine you are in a massive library. You are holding a specific book (the `current token`) and you want to write a summary note about it. 
To do this effectively, you need to find related information from other books on the shelf.

    - The **Query ($Q$)**: The Sticky Note Question<br/>
        Analogy: You stick a post-it note on the book you are holding. On it, you write what you are looking for relative to this book.<br/>
        Example: If you are holding a book about "The Empire State Building," your Query might be: "Find me books about architecture or New York history".<br/>
        In Transformers: The Query is a vector representing the current token's intent. It isn't the word itself, but what the word is looking for to complete its meaning.

    - The **Key ($K$)**: The Dewey Decimal / Spine Label<br/>
        Analogy: Every book on the shelf has a label on its spine. This label advertises what the book is about to anyone looking.<br/>
        Example: A book about "Concrete Mixing" might have a Key label: "Construction materials, Engineering."<br/>
        In Transformers: The Key is a vector representing what a token defines itself as. It is used solely for matching purposes.

    - The **Attention Score**: Matching Q and K<br/>
        Analogy: You walk down the aisle comparing your Query (Sticky note) to every book's Key (Spine label).<br/>
        The Match: If your note says "Architecture" and a spine says "Cooking," the match is 0% (ignore).<br/> 
        If the spine says "Skyscrapers," the match is 95% (pay close attention).

    - The **Value ($V$)**: The Content of the Book
        Analogy: Once you find a match, you don't just stare at the spine (Key); <br/>
        you open the book and read the content (Value).<br/>
        You only take as much content as the match score dictates. If the match was 95%, you read a lot of that book. If it was 5%, you barely skim a sentence.<br/>
        In Transformers: The Value is the actual informational content of the token that will be passed along to the next layer.

- Current vs. Previous Tokens roles

    Now, let's translate this to the mechanics of the Transformer and how it processes a sentence like:

    > "The river flows to the bank"

    When the model processes the word `bank` (Current Token):

    1. Role of the **Query (Current Token)**:<br/>
    The model projects the embedding for "bank" into a Query vector.<br/>
    This Query effectively asks: "I am the word 'bank'. To know if I mean a financial building or land alongside water, I need to see if there are words like 'money' or 'river' nearby."

    2. Role of the **Keys (All Tokens)**:<br/>
    The model looks at `The`, `river`, `flows`, `to`, `the`, and `bank`. It projects all of them (including "bank" itself) into Key vectors.<br/>
    The Key for "river" effectively screams: "I am a nature word! I am about water!"

    3. The Interaction (**Dot Product**):<br/>
    The Query of `bank` multiplies with the Key of `river`.<br/>
    Because "water" contexts answer "bank's" question, the resulting score is `High`.<br/>
    The Query of "bank" also multiplies with the Key of "The" and so on. The score is likely `Low` (less relevant).

    4. Role of the **Value (All Tokens)**:<br/>
    The model then takes the `Value` vector of `river`.<br/>
    Because the score was high, the model "absorbs" a large amount of the mathematical meaning from `river` and adds it to `bank`.<br/>
    Result: The representation of `bank` is updated. It is no longer just `bank`; it is now `bank (context: river/nature)`.

    > "I went to the bank to deposit money."

    When processing `money` here, the Query will interact strongly with tokens like `deposit` and `bank`, pulling in financial context instead, and it reinforces the financial meaning of `bank`. <br/>But here bank is not agnostic; it was already strongly influenced previously by `deposit`.<br/>
    _This is the magic of stacked layers and residual connections_

[📺Why the name Query, Key and Value?](https://www.youtube.com/watch?v=viCl2T7vx64)

>Attention answers the question of the current token by looking up the best matching indexes from past tokens and gathering their content.

#### Self-attention: relevance scoring
The relevance score between the current token and each previous token is computed
by taking the `dot product` of the `query vector` of the current token with the `key vectors` of all tokens.
This results in a set of scores that indicate how much attention the current token should pay to each previous token.
Passing that by a `softmax` operation normalizes these scores so they `sum up to 1`.

![image-2.png](./39fc787c_image-2.png)

#### Self-attention: combining information
The relevance scores are then normalized using a softmax function to produce attention weights.
These weights are used to compute a `weighted sum` of the `value vectors` of all tokens (more relevant tokens contribute more to the sum, proportionally to their scores),
resulting in a new representation for the current token that incorporates relevant information from the context.

![image-3.png](./39fc787c_image-3.png)

> $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$ <br/>
    Where: <br/>
    $Q$ = Query matrix, $K$ = Key matrix, $V$ = Value matrix,  $d_k$ = dimensionality of the key vectors

- Pseudo-code for a single attention head:

    ```python
    import numpy as np

    def attention_head(x, W_Q, W_K, W_V, mask=None):
        """
        x: Input tokens (Batch, Seq_Len, Dim)
        """
        d_k = W_Q.shape[-1] # Dimension of the keys (for scaling)

        # 1. PROJECTION
        # Create the "Sticky Note" (Q), "Spine Label" (K), and "Book Content" (V)
        Q = x @ W_Q 
        K = x @ W_K 
        V = x @ W_V 

        # 2. SCORING (The "Library Search")
        # Match every Query with every Key.
        # We swap the last two axes of K to allow matrix multiplication.
        # Result shape: (Batch, Seq_Len, Seq_Len) -> The "Attention Map"
        scores = Q @ K.swapaxes(-1, -2) / np.sqrt(d_k)

        # 3. MASK: apply mask if provided for padding or future tokens (anti-cheating)
        # Look-Ahead Masking for Causal Attention
        # 0 allowed: (e.g., Word 3 can look at Word 1 and 2)0 allowed
        # -inf (-1e9) blocked: (e.g., Word 1 cannot look at Word 2 and 3)
        # During training, x is typically the entire sentence (all tokens at once), not just the last one.
        # While "generating text" is writing one word after another (sequential), transformers are designed to be parallel.
        # The attention mechanism processes all tokens simultaneously, so masking is needed to prevent future token information leakage for tokens that come earlier in the sequence.
        if mask is not None:
            scores = np.where(mask, scores, -1e9)  # Apply mask to scores

        # 4. NORMALIZATION
        # Turn raw scores into probabilities (percentages that sum to 100%)
        weights = softmax(scores, axis=-1)

        # 4. AGGREGATION (The "Reading")
        # Create a new representation by summing Values weighted by their relevance.
        output = weights @ V 

        return output            
    ```

### Look-Ahead Masking for Causal Attention

In causal attention, each token can only attend to previous tokens and itself, not future tokens.<br/>
This is crucial for tasks like language modeling, where the model generates text sequentially.<br/>
To implement `causal attention`, the `look-ahead mask` prevents the model from seeing future tokens. <br/>
This mask is typically a triangular matrix that allows attention to the current and previous tokens while blocking future tokens.<br/>
<br/>
During text generation (GPT-style), future tokens literally do not exist yet.<br/>
However, in the `Prompt Phase` (when we feed the model an existing sentence to start it off), all prompt tokens are co-present.<br/>
So why do we still ignore it?<br/>
<br/>
Here is the breakdown of why we "blindfold" the model even when the text is right there.<br/>

1. The **Writer** Mode (Generation)<br/>
    Scenario: The model is generating a new story. It has written 3 words ("The bank of").<br/>
    Current State: It is trying to predict the next token.<br/>
    This is not "hidden"—it hasn't been born yet. It is 0.00% existent.<br/>
    Conclusion: In this mode, the mask isn't a rule; it’s a physical reality. <br/>

2. The **Prompt** Mode (The Counter-Intuitive Part)<br/>
    Scenario: You give the model a completed sentence as a prompt: "The bank of the river is deep" (7 words). <br/>
    You want it to continue the story from there.<br/>
    Why not let 'bank' (word 2) look at 'river' (word 5) so it knows it's a river bank immediately?"<br/>
    The Problem (Brain Consistency):<br/>
    During training, the model learned that the vector for "bank" must represent "I am 'bank' and I don't know what comes next."<br/>
    If you suddenly let "bank" peek at "river" during the prompt phase, the vector for "bank" changes. <br/>
    It becomes "I am 'bank' and I know 'river' is coming."<br/>
    The Crash: When the model tries to predict the next word, it uses its learned weights. Those weights expect the "ignorant" version of bank. If you feed them the "all-knowing" version of bank, the math breaks down. The model enters a state it never practiced, and the output becomes inconsistent.

3. The **Editor** Mode<br/>
    Different type of Transformer (`Encoder-only`), where the goal is not to generate text (e.g., classify sentiment or extract entities), are a **Bi-directional** Transformer (like BERT): every token can see previous and future tokens.<br/>
    Analogy:GPT (Causal): A writer. They write page 3. They have no idea what happens on page 10 because they haven't written it yet.<br/>
    BERT (Bi-directional): An editor. They have the finished manuscript. They read page 3, see a vague reference, flip to page 10 to understand it, and then go back to fix the note on page 

---    

Summary Table

| Logic   | GPT (Decoder-only)   | BERT (Encoder-only) |
|---------|-----------------|----------------|
| Goal    | Write the next word.    | Understand the existing text. |
| Can Word 3 see Word 7?  | No. | Yes. |
| Why?    | Because in generation, Word 7 doesn't exist. In prompts, we must pretend it doesn't to stay consistent. | Because the text is already finished, so looking ahead helps "enrich" the meaning. |

---

> GPT is designed to live in the moment. BERT is designed to analyze history.

## Residual Connections and Normalization

In each Transformer block, around "Attention" and "Feed Forward.", **Residual Connections** and **Normalization** are the unsung heroes.

Without these two components, a Transformer would be nearly impossible to train.


| Original paper diagram | Block detail | Self-attention detail |
|:--- | :--- | :--- |
|![image.png](./cd7d10aa_image.png) |![image-2.png](./cd7d10aa_image-2.png) |![image-3.png](./cd7d10aa_image-3.png) |

---

### 1. Residual Connections (The "Skip")

Also known as **Skip Connections**, these are the pathways that allow data to "skip" a processing layer and be added back in later.

* **Role:** Their primary job is to solve the **Vanishing Gradient Problem**. In deep networks, as the model learns (during backpropagation), the error signal gets weaker and weaker as it travels back from the output to the input. If the signal vanishes, the early layers of the network stop learning. Residual connections provide a "superhighway" for this signal to flow uninterrupted.
* **Context:** In a Transformer block, the input $x$ goes into a sub-layer (like Attention). The output of that layer is not just passed to the next step; it is mathematically added to the original input.
    * *Equation:* $\text{Output} = \text{Layer}(x) + x$
* **Why they are important:** They allow Transformers to be stacked very deeply (dozens or hundreds of layers). Without them, the network would forget the original input features (like the actual words in the sentence) after just a few layers of processing.

#### Analogy for Residual Connections: "The Editor and the Original Draft"

Imagine a writer (the Input) gives a draft of a chapter to an Editor (the Attention Layer).

* **Without Residual Connections:** The Editor rewrites the whole chapter and passes *only* the new version to the next Editor. After 10 editors, the original plot might be completely lost because every editor changed it slightly.
* **With Residual Connections:** The Editor writes their suggested changes on a sticky note, but **staples the sticky note to the original page**. They pass *both* (Original + Changes) to the next person.
* **The Benefit:** Even if the Editor does a bad job (or the layer hasn't learned anything yet), the next person still has the original draft intact. The information is preserved, and the "original voice" is never lost.

---

### 2. Layer Normalization (The "Stabilizer")

Normalization ensures that the numbers (activations) flowing through the network stay within a reasonable range (usually with a mean of 0 and a variance of 1). Transformers specifically use **Layer Normalization (LayerNorm)**.

* **Role:** It stabilizes the training process. Without normalization, the values inside the network can vary wildly (some neurons outputting 0.001, others outputting 10,000). This makes the optimizer's job a nightmare, leading to "exploding" gradients or training that never converges.
* **Context:** It is applied immediately after the residual connection (in the original design). It looks at all the hidden features of a *single* word (token) and normalizes them.
    * *Equation:* For a given input vector $x$ with features $x_1, x_2, ..., x_n$:
      1. Compute the mean: $\mu = \frac{1}{n} \sum_{i=1}^{n} x_i$
      2. Compute the variance: $\sigma^2 = \frac{1}{n} \sum_{i=1}^{n} (x_i - \mu)^2$
      3. Normalize each feature: $\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$ (where $\epsilon$ is a small constant to prevent division by zero)
      4. Scale and shift: $y_i = \gamma \hat{x}_i + \beta$ (where $\gamma$ and $\beta$ are learnable parameters)
    * *Note:* This is different from **Batch Normalization** (used in image processing), which normalizes across different images. LayerNorm normalizes across the features of a single input, which is crucial for text because sentences have different lengths and batch statistics can be noisy.
* **Why they are important:** It allows the model to learn much faster and with higher learning rates. It ensures that no single feature dominates the calculation simply because it happens to have a larger numerical scale.

#### Analogy for Layer Normalization: "The Sound Engineer"

Imagine a choir where every singer (neuron) has a microphone.

* **The Problem:** Some singers are naturally whispering (low values), and some are screaming (high values). If you try to mix this audio, the screaming will drown out the details of the whisperers.
* **The Normalization Role:** You place a sound engineer at the mixing board. Before the sound goes to the audience (the next layer), the engineer adjusts the gain on every microphone so that everyone is singing at roughly the same volume.
* **The Benefit:** Now, the audience can hear the *harmony* and the *content* of the song, rather than just hearing the loudest person. It ensures the "loudness" doesn't distract from the "meaning."

---

### 3. Application: The "Add & Norm" Block

In the architecture, these two are almost always used together as a unit called the **Add & Norm** block.

1.  **Add:** The model takes the output of the Attention mechanism and *adds* it to the original input. (Preserves information).
2.  **Norm:** The model takes that combined result and *normalizes* it. (Stabilizes the scale).

This cycle repeats for every single sub-layer in the Transformer block.

---

### Summary Table

| Component | Scientific Role | The "Human" Role | Key Benefit |
| :--- | :--- | :--- | :--- |
| **Residual Connection** | Gradient flow, Identity mapping | **The Memory Keeper** | Prevents the model from forgetting the input; allows deep stacking. |
| **Layer Normalization** | Feature scaling, variance reduction | **The Equalizer** | Prevents values from spiraling out of control; speeds up training. |



### Attention diagram

![image.png](./59d035fb_image.png)

<sub>`h`: multiple attention heads running in parallel</sub>
